# Proyecto 3 | Computer Vision 2025

### Renatto Guzmán y Adrian Fulladolsa

In [12]:
## Imports

import cv2
import numpy as np


In [6]:
# Read images

img1 = cv2.imread('input/1.jpg')
img2 = cv2.imread('input/2.jpg')
img3 = cv2.imread('input/3.jpg')

In [ ]:
# Elegir una imagen base

base_img = img2

### Crear homografías

In [11]:
# Establecer un mapa de correspondencia entre las imágenes
# (1, 2) -> (2, 3) -> (3, 1)


# Crear el detector ORB
orb = cv2.ORB_create(nfeatures=2000)

# Detectar puntos clave y descriptores
kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)
kp3, des3 = orb.detectAndCompute(img3, None)


bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

# Matching img1 <-> img2
matches_12 = bf.match(des1, des2)
matches_12 = sorted(matches_12, key=lambda x: x.distance)

# Matching img3 <-> img2
matches_32 = bf.match(des3, des2)
matches_32 = sorted(matches_32, key=lambda x: x.distance)


In [13]:
# Calcular la homografía entre las imágenes


# Función para obtener homografía entre dos imágenes
def get_homography(kpA, kpB, matches):
    ptsA = np.float32([kpA[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    ptsB = np.float32([kpB[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(ptsA, ptsB, cv2.RANSAC)
    return H, mask


In [14]:
H_12, mask_12 = get_homography(kp1, kp2, matches_12)
H_32, mask_32 = get_homography(kp3, kp2, matches_32)


In [15]:
H_12

array([[ 1.46793056e+00, -4.25353458e-02, -5.99041832e+02],
       [ 2.04802819e-01,  1.27092683e+00, -2.00980097e+02],
       [ 3.01685356e-04, -1.25708002e-05,  1.00000000e+00]])

### Hacer warp

In [16]:
# Tamaños de las imágenes
h1, w1 = img1.shape[:2]
h2, w2 = img2.shape[:2]
h3, w3 = img3.shape[:2]

# Coordenadas de las esquinas de cada imagen
corners1 = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
corners2 = np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)
corners3 = np.float32([[0, 0], [0, h3], [w3, h3], [w3, 0]]).reshape(-1, 1, 2)

# Transformar esquinas a la vista de la imagen base
warped_corners1 = cv2.perspectiveTransform(corners1, H_12)
warped_corners3 = cv2.perspectiveTransform(corners3, H_32)

# Concatenar todas las esquinas para calcular la extensión del canvas
all_corners = np.concatenate((warped_corners1, corners2, warped_corners3), axis=0)

# Obtener los extremos del canvas
[xmin, ymin] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
[xmax, ymax] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

# Traslación para evitar coordenadas negativas
translate = [-xmin, -ymin]
T = np.array([[1, 0, translate[0]], [0, 1, translate[1]], [0, 0, 1]])  # matriz de traslación

# Tamaño del canvas final
canvas_width = xmax - xmin
canvas_height = ymax - ymin

# Warp de cada imagen al canvas
result = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8)

# Warp de img1
warp1 = cv2.warpPerspective(img1, T @ H_12, (canvas_width, canvas_height))

# Warp de img3
warp3 = cv2.warpPerspective(img3, T @ H_32, (canvas_width, canvas_height))

# Pegar img2 (base) directamente en el canvas
result[translate[1]:translate[1]+h2, translate[0]:translate[0]+w2] = img2

# Combinar los warps (img1 y img3) encima del resultado base
mask1 = (warp1 > 0)
result[mask1] = warp1[mask1]

mask3 = (warp3 > 0)
result[mask3] = warp3[mask3]

# Mostrar o guardar el resultado
cv2.imshow('Panorama', result)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [17]:
cv2.imwrite('output/panorama.jpg', result)

True

### Utilizar blending

In [18]:
def warp_and_blend(img, H, canvas_size, T, weight_acc, image_acc):
    h, w = img.shape[:2]
    
    # Warp de imagen
    warped_img = cv2.warpPerspective(img, T @ H, canvas_size)
    
    # Crear una máscara blanca del tamaño de la imagen original
    mask = np.ones((h, w), dtype=np.uint8) * 255
    warped_mask = cv2.warpPerspective(mask, T @ H, canvas_size)
    
    # Crear una máscara suave (feathered) difuminando bordes
    kernel_size = 51  # cuanto más grande, más suave el borde
    soft_mask = cv2.GaussianBlur(warped_mask.astype(np.float32), (kernel_size, kernel_size), 0)
    soft_mask = soft_mask / 255.0  # Normalizar entre 0 y 1
    soft_mask = np.expand_dims(soft_mask, axis=2)  # Convertir a shape (H, W, 1)
    
    # Acumular imagen ponderada y pesos
    image_acc += warped_img.astype(np.float32) * soft_mask
    weight_acc += soft_mask


In [20]:
# Canvas size
canvas_size = (canvas_width, canvas_height)

# Inicializar acumuladores
image_acc = np.zeros((canvas_height, canvas_width, 3), dtype=np.float32)
weight_acc = np.zeros((canvas_height, canvas_width, 1), dtype=np.float32)

# Warp y blend de img1, img2, img3
warp_and_blend(img1, H_12, canvas_size, T, weight_acc, image_acc)
warp_and_blend(img2, np.eye(3), canvas_size, T, weight_acc, image_acc)
warp_and_blend(img3, H_32, canvas_size, T, weight_acc, image_acc)

# Evitar división por cero
weight_acc[weight_acc == 0] = 1

# Resultado final normalizado
blended_result = (image_acc / weight_acc).astype(np.uint8)

# Mostrar o guardar
cv2.imshow('Blended Panorama', blended_result)
cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.imwrite('output/blended_panorama.jpg', blended_result)


True